# Настройка инструментов

<div class="border" style="border: 2px solid black;width: 99%;height: auto;margin-top: 20px;border-bottom: none;"> <p class="legend" style="margin-top: -8px;margin-left: 10px;padding-left: 15px;background: white;width: 209px;font-weight: 600;">Инициализация среды</p></div>


*Импортируем библиотеки для работы с данными.*


In [1]:
import pandas as pd

*Загрузим конфигурации research обалсти.*

In [2]:
from settings import CONFIGS_DIR

import json

with open(CONFIGS_DIR / 'baseline_v1.json', encoding="utf-8") as f:
    config = json.load(f)

Инструменты пайплайна

In [3]:
from research.preprocessing import SchemaNormalizationTransformer
from research.preprocessing.surface_processing import SurfaceProcessingTransformer
from research.preprocessing.advanced_processing.transformer import AdvancedProcessingTransformer

<div class="border" style="border: 2px solid black;width: 99%;height: 10px;border-top: none;"></div>


<div class="border" style="border: 2px solid black;width: 99%;height: auto;margin-top: 20px;border-bottom: none;"> <p class="legend" style="margin-top: -8px;margin-left: 10px;padding-left: 15px;background: white;width: 319px;font-weight: 600;">Загрузка и настройка ресурсов</p></div>


**Наименование:** titanic.<br>**Путь к файлу:** datasets/raw/train.csv <br><br>


_Считаем и переименуем._

In [4]:
from settings import RAW_DATASET_DIR

titanic = pd.read_csv(
    RAW_DATASET_DIR / config["raw_datasets"]["train"]
)

titanic = SchemaNormalizationTransformer().fit_transform(titanic)

titanic.head(0)

,passenger_id,ticket_id,name,sex,age,same_importance_relatives,high_importance_relatives,category_significance,fare,cabin,embarked,survived


<div class="border" style="border: 2px solid black;width: 99%;height: 10px;border-top: none;"></div>


# Предобработка данных


In [5]:
titanic = SurfaceProcessingTransformer().fit_transform(titanic)

# Обобщение выводов предобработки


<ul>
<li>

**@titanic `ticket_id`.**

Данные в этом признаке содержали аномалию, связанную с случайными символами "." и "/", но проведя исследование аномалии мы пришли к выводу, что проводить коррекцию данных не имеет смысла.

Некоторые люди с одной фамилией или иными признаками родства имели разные номера билетов, например билеты "2695", "2690".

Можно попробовать разделить признак ticket_id на id и строчную преписку из букв;
</li>

<li>

**@titanic `name`.**

Обозначение осознанности, которое несёт смысл вставлено в имя (Master, Mr, Miss, Mrs), когда лучше его вынести в отдельный признак на равне с признаком age. Создать признак age_level.

 поле name заложены имена и фамилии. Из-за имени дубликатов практически нет, что ломает нам всю картину для более глубокого анализа, следовательно нужно попробовать создать признак  family.
</li>

<li>

**@titanic `sex`.**

По какой-то причине на титанике было 577 человек мужского пола, в то время как женского пола всего 313.

Также пока не понятно что делать с пропусками. Нужно смотреть корреляции.

</li>

<li>

**@titanic `age`.**

Нужно создать новые признаки, объединив признак age с другими.

Подозрения на аномалии:
- Очень большое количество пропусков в признаке age связано с пассажирами 3ей категории;
- Большое количество детей меньше года;
- 77% пропущенных значений - люди из 3ей категории;
- 5 человек старше 70.
<br>

</li>

<li>

**@titanic `fare`.**

Люди, которые заплатили сверх цену платили за несколько кабин. Одна и та же цена  записывается каждому члену семьи.

Также было обнаружено, что есть определённые категории цен, но при этом сами по себе они сильно разнятся - нужно посмотреть под одну категорию попадают только члены одной семьи или в целом многие заплатили одинаково. Возможно сегментация может что-то прояснить в данных, чего мы не видели. (очень часто все дубликаты имеют одну категорию значимости, а при увеличении стоимости билета увеличиваются также и количества родственников, которых оформили на один билет)

Есть пассажиры, которые заплатили 0. Причина также не ясна.

Есть несколько закономерностей, которые связывают категорию людей не оплативших проезд:

1. Абсолютно все эти люди мужчины;
2. Каждый мужчина не оплативший проезд не имел никаких родственников;
3. Все сели в одном и тот же порту;
4. Все ехали либо по одиночке либо вместе с другими такими же
<br>
Мы могли бы счесть этих пассажиров важными, если бы в датасете не было ещё 500 человек абсолютно без родственников. То есть первые 3 признака буквально не говорят ни о чём, а 4ый как будто просто следствие из остальных.

В общем, в данных просто не хватает признаков, чтобы описать эту отдельную категорию людей, которые абсолютно не заплатили за проезд. Я склоняюсь к тому, чтобы заполнить эти нулевые значения. В любом случае их слишком мало, чтобы очень сильно углублятся в исследование - нужно иметь в виду.

</li>

<li>

**@titanic `cabin`.**

Удалить признак cabin, превратив его в набор признаков:
   - палуба;
   - признак неопределённости: 1 → конкретная каюта известна, 2+ -> пассажир связан с несколькими каютами;
   - каюта определенна однозначно;
   - члены семьи находятся далеко друг от друга на этаже;
   - один из членов семьи находится в конце корридора.
   - создать признак в одной кабине было больше одного человека (тут есть вероятность, что они повторяются даже без родства)
</li>
</ul>

<div class="border" style="border: 2px solid black;width: 99%;height: 10px;border-top: none;"></div>


# Решение по выводам из EDA


<div style="
color: green;
padding: 20px;
border: 2px dotted green;
">

_Были изучены:_

* _Исходные признаки Titanic;_
* _Признаки, которые были добавлены после поверхностного знакомства с данными;_
* _Признаки, которые показали высокую информативность после иерархического анализа зависимостей._

_Основная цель заключалась в поиске признаков, связанных с целевой переменной `survived`. Проведённый анализ показал, что большинство наблюдаемых зависимостей нельзя интерпретировать напрямую. Итоговая формула `survived` объясняется существованием небольшого числа скрытых факторов, которые также объясняют большую часть обнаруженных корреляций в принципе._

**Лучше всего в нашей ситуации работают признаки с таким смыслом:**

* _**Семейная структура.** Признаки семейной структуры содержат полезную информацию, однако значительная её часть пересекается с признаками `sex` и `age`: максимальный эффект у семей из двух человек `family_group`,но основным фактором является наличие женщин (группы только мужчин имеют минимальную выживаемость, смешанные группы занимают промежуточное положение, а группы с женщинами — наиболее высокое) Поэтому при построении модели необходимо оценить, какие признаки семейной структуры (`family_size`, `family_size_group`, `is_alone`, `high_importance_relatives` и др.) действительно повышают качество модели, а какие лишь дублируют уже существующие зависимости.
* _**Cоциальный класс.** Наиболее информативным оказался `category_significance`, который включает в себя информацию о классе обслуживания и положении пассажира на корабле (всё связанное с `cabin`, `embarket`). Дополнительные признаки, связанные с каютами и палубами, в основном отражают тот же фактор._
* **Демографические признаки.** `sex` и `age` остаются базовыми признаками, которые необходимо учитывать при построении модели.
* _**Размер группы по билету.** `ticket_group_size` содержит информацию о совместном путешествии пассажиров и может использоваться как отдельный признак, так как описывает социальные связи, которые не всегда совпадают с семейными._

Остальные признаки требуют проверки на избыточность и вклад в качество модели.

</div>

<div style="
color: green;
padding: 20px;
border: 2px dotted green;
">

_**Второстепенная цель - заполнение пропусков.**_

_Заполнение `deck` и `cabin` через любые существующие корреляции (`sex`, `age`, `fare`, `category_significance`, `embarket`, `ticket_prefix`) оказалось невозможным. Более того, по мере анализа всех существующих корреляций, каждый раз выяснялось, что их существование было следствием аномалии (особенностью заполнения данных): информация о каютах сохранилась преимущественно для пассажиров первого класса._

_Стало понятно, что высокая связь с целевым признаком достигалась за счёт дублирования информации `category_significance`, так как данный признак полностью включал в себя информацию признака `deck`. Соответственно в Titanic все наиболее значимые для выживания привилегии связанные с кабинами и палубами получали пассажиры первого класса._

_Исследование стоимости билетов `fare` показало, что `cabin_count` количество кабин помогало сильнее восстановить различия внутри отдельных групп. К сожалению, восстановить данный признак тоже просто не оказалось возможным;_

_Исследование признака age показало, что отдельные признаки (`social_role`, `social_role_and_sex`, `category_significance`) недостаточно хорошо описывают возраст пассажиров. Наиболее точное восстановление удалось получить при совместном использовании признаков `category_significance`, `family_type` и `social_role`, которые одновременно учитывают социальное положение пассажира, семейную структуру и жизненный этап. Заполнение пропусков медианным возрастом внутри таких групп показало наименьшую среднюю абсолютную ошибку (MAE ≈ 7.55 года) по сравнению с другими рассмотренными стратегиями группировки. Данный подход выбран в качестве основной стратегии заполнения пропусков возраста._

</div>


<div style="
color: green;
padding: 20px;
border: 2px dotted green;
">

_**Второстепенная цель - уменьшить шум.**_

_В ходе анализа было установлено, что ряд признаков описывает одни и те же скрытые свойства пассажиров с разных сторон. Такие признаки потенциально могут конкурировать между собой при обучении модели и не всегда дают дополнительную информацию при совместном использовании._

_Признаки `same_importance_relatives`, `high_importance_relatives`, `ticket_group_size`, `is_alone` и `family_size` характеризуют наличие и размер группы, в которой путешествовал пассажир. При этом было отмечено всего 44 случая путешествия с людьми, чьи связи с группой оставались неизвестными. В большинстве таких случаев неизвестные пассажиры лишь дополняли семейную группу одним-двумя участниками._

_Признаки `female_count`, `male_count`, `child_count` и `family_type` описывают состав семейной группы. При этом `family_type` является обобщённым представлением состава группы и потенциально может дублировать информацию отдельных счётчиков._

_Признаки `age`, `social_role` и `social_role_and_sex` описывают жизненный этап пассажира с разной степенью детализации. Исследование восстановления возраста также показало, что семейная структура частично связана с этим же фактором, поэтому `family_type` может конкурировать с признаками жизненного этапа._

_Признаки `fare`, `category_significance`, `deck`, `cabin_assignment` и `ticket_prefix` частично описывают социальное положение и уровень обслуживания пассажира. Было установлено, что `category_significance` включает информацию о `deck`, а большинство значений `ticket_prefix` также практически полностью привязано к определённым категориям обслуживания. Поэтому их самостоятельная информативность требует дополнительной проверки._

_Анализ `embarked` показал, что его обнаруженные зависимости преимущественно объясняются различным распределением категорий обслуживания между портами посадки. После учёта социального класса самостоятельная информативность порта существенно снижается. Принято решение не использовать данный признак как основной кандидат для модели._

_Необходимо провести анализ важности признаков._

</div>

<div style="
color: green;
padding: 20px;
border: 2px dotted green;
">

_**Второстепенная цель - улучшение качества признаков и создание новых.**_

_В ходе анализа проводилось агрегирование пассажиров по ticket_number и surname, что позволило получить признаки, описывающие не отдельного пассажира, а структуру группы, в которой он путешествовал._

_Группировка по `ticket_number` показала, что количество пассажиров (`passenger_count`) связано с распределением `fare`, `ticket_prefix`, `deck`, `category_significance` и `embarked`. Один билет часто объединял пассажиров с общими характеристиками обслуживания и размещения. При этом были обнаружены случаи, когда один билет объединял несколько семей: дополнительные пассажиры в таких группах чаще являлись попутчиками, а не родственниками. Это позволило рассматривать `ticket_group_size` не только как размер группы, но и как отдельный показатель совместного путешествия._

_Группировка по surname позволила выделить семейные группы и создать `family_size`, `family_type`, `female_count`, `male_count` и `child_count`. Исследование показало, что одного размера семьи недостаточно для описания семейного фактора: группы одинакового размера могли иметь существенно различный состав. Поэтому состав группы рассматривается как отдельная характеристика, а `family_size` и `family_type` — как основные кандидаты для дальнейшей проверки в модели._

Анализ `deck`, `cabin` и `category_significance` показал, что часть наблюдаемых закономерностей нельзя напрямую интерпретировать как влияние расположения каюты. Заполнение `deck` сохранилось преимущественно для пассажиров первой категории, поэтому необычные зависимости отдельных палуб с полом, возрастом и другими признаками могут отражать состав размеченной выборки, а не свойства самой палубы._

_Например, наблюдение о преобладании мужчин на отдельных палубах не позволяет делать общий вывод о связи пола с расположением на корабле. При этом `category_significance` оказался более устойчивым способом представить информацию о социальном положении пассажира, поскольку он сохраняется для значительно большей части наблюдений.

_Аналогичный эффект был обнаружен для `embarked`. Различия между портами в основном объяснялись различным распределением категорий обслуживания: в Q преобладали пассажиры третьей категории, в C была значительно выше доля первой категории, а S имел наиболее смешанный состав. После учёта категории обслуживания самостоятельная связь порта с выживаемостью существенно снижалась. Поэтому создание новых признаков на основе `embarked` не имеет очевидного преимущества._

_Исследование возраста показало, что существующие признаки позволяют уточнить его представление без создания большого количества дополнительных категорий. Наиболее точное восстановление пропущенного `age` получено при совместном использовании `category_significance`, `family_type` и `social_role` (MAE ≈ 7.55 года). Это показало, что социальное положение, семейная структура и жизненный этап совместно дают более точное описание возраста, чем отдельные признаки._

_Таким образом, feature engineering в данном исследовании позволил перейти от отдельных характеристик пассажира к нескольким скрытым факторам: семейной структуре, структуре группы по билету, социальному положению и жизненному этапу._

</div>

# Углубленная обработка


In [6]:
titanic.info()

<class 'pandas.DataFrame'>
Index: 887 entries, 0 to 890
Data columns (total 12 columns):
 #   Column                     Non-Null Count  Dtype   
---  ------                     --------------  -----   
 0   passenger_id               887 non-null    uint16  
 1   ticket_id                  887 non-null    str     
 2   name                       887 non-null    str     
 3   sex                        887 non-null    category
 4   age                        710 non-null    float32 
 5   same_importance_relatives  887 non-null    uint8   
 6   high_importance_relatives  887 non-null    uint8   
 7   category_significance      887 non-null    category
 8   fare                       887 non-null    float64 
 9   cabin                      204 non-null    str     
 10  embarked                   887 non-null    category
 11  survived                   887 non-null    int8    
dtypes: category(3), float32(1), float64(1), int8(1), str(3), uint16(1), uint8(2)
memory usage: 75.3 KB


## DF @family_group

<div class="border" style="border: 2px solid black;width: 99%;height: auto;margin-top: 20px;border-bottom: none;"> <p class="legend" style="margin-top: -8px;margin-left: 10px;padding-left: 15px;background: white;width: 180px;font-weight: 600;">Добавление признака</p></div>


_Анализ показал, что одного размера семьи недостаточно для описания семейного фактора: группы одинакового размера могли иметь различный состав, а различия в составе были связаны с существенно разной выживаемостью. Поэтому в итоговую таблицу добавляются следующие признаки:_

- _`family_size` — размер семейной группы. Позволяет учитывать сам факт наличия и размер семьи, а также различия между одиночными пассажирами и пассажирами, путешествующими с родственниками;_
- _`female_count` — количество женщин в семейной группе. Наличие женщин оказалось одним из наиболее выраженных факторов, связанных с выживаемостью семейных групп;_
- _`male_count` — количество мужчин в семейной группе. Позволяет учитывать состав семьи и отличать группы преимущественно мужского состава от смешанных;_
- _`child_count` — количество детей в семейной группе. Позволяет учитывать наличие детей и дополнительно описывает возрастной состав семьи. Также замечен перекос среди в сторону пассажиров с детьми каждого пола;_
- _`family_type` — обобщённый тип семейной группы. Объединяет информацию о наличии мужчин, женщин и детей и позволяет различать группы с одинаковым размером, но разным составом._

_При этом `age_mean`, `age_min`, `age_max`, `fare`, `ticket_count` и другие агрегаты семейной группы не добавляются, поскольку в ходе исследования не показали достаточного преимущества перед уже существующими признаками или потенциально дублируют информацию, представленную другими факторами._

In [7]:
titanic = AdvancedProcessingTransformer.create_family_features(titanic)

<div class="border" style="border: 2px solid black;width: 99%;height: 10px;border-top: none;"></div>


## DF @titanic `cabin`


<div class="border" style="border: 2px solid black;width: 99%;height: auto;margin-top: 20px;border-bottom: none;"> <p class="legend" style="margin-top: -8px;margin-left: 10px;padding-left: 15px;background: white;width: 220px;font-weight: 600;">Устранение пропусков</p></div>


_В ходе исследования были рассмотрены различные способы восстановления признаков `deck` и `cabin`. Существенных закономерностей, позволяющих достоверно восстановить пропущенные значения, обнаружить не удалось._

_Кроме того, анализ корреляций показал, что дополнительной информации, которая могла бы повысить качество модели, выявлено не было, и, что информация, содержащаяся в признаках `deck` и `cabin`, в значительной степени уже отражается другими признаками:_
* _`deck` ↔ `category_significance` = 0.84_
* _`deck` ↔ `ticket_prefix` = 0.63_
* _`deck` ↔ `cabin_assignment` = 0.57_
* _`deck` ↔ `age_group` = 0.41_
* _`deck` ↔ `social_role_and_sex` = 0.42_

_Корреляция признака `deck` 0.27 в сравнении с 0.2 для `category_significance`. Если бы был выбор оставить только один признак, то `deck` даже предпочтительнее, но мы строим модель из нескольких признаков._

_Принято решение отказаться от использования признаков `deck` и `cabin`, оставив только производные признаки, содержащие извлекаемую из них информацию._

In [8]:
titanic = AdvancedProcessingTransformer.remove_cabin_features(titanic)

<div class="border" style="border: 2px solid black;width: 99%;height: 10px;border-top: none;"></div>


## DF @titanic `ticket_group_size`


<div class="border" style="border: 2px solid black;width: 99%;height: auto;margin-top: 20px;border-bottom: none;"> <p class="legend" style="margin-top: -8px;margin-left: 10px;padding-left: 15px;background: white;width: 180px;font-weight: 600;">Добавление признака</p></div>


_Анализ показал, что один билет мог объединять не только родственников, но и других пассажиров, путешествующих совместно. Поэтому `ticket_group_size` рассматривается как отдельный показатель размера группы совместного путешествия._

In [9]:
titanic = AdvancedProcessingTransformer.create_ticket_group_size(titanic)

<div class="border" style="border: 2px solid black;width: 99%;height: 10px;border-top: none;"></div>


_Теоретически можно поделить `fare` на этот признак, чтобы сместить распределение, но тогда признаки будут содержать друг друга и мы не сомжем нормально передавать `ticket_group_size` в модель._

_На мой взгляд, конечно, сам признак достаточно мусорный, он только путает и содержит большую часть информации размера семьи. Но пока что-то делать с ним я не хочу._

## DF @titanic `sex`


_Сохраняется как один из наиболее выраженных факторов выживаемости. Анализ показал существенное различие между мужчинами и женщинами, причём этот фактор проявлялся и внутри семейных групп. Поэтому семейные признаки не заменяют `sex`, а дополняют его._

## DF @titanic `category_significance`


_Сохраняется как наиболее устойчивое представление социального положения пассажира. В отличие от `deck`, `cabin` и связанных с ними признаков, этот признак сохраняет информацию о категории обслуживания для значительно большей части наблюдений и при этом включает существенную часть информации, связанной с размещением пассажира._

## DF @titanic `fare`.

_Сохраняется, поскольку стоимость билета отражает различия внутри социальных категорий, которые не полностью сводятся к `category_significance`. В ходе анализа также было установлено, что `fare` позволяет выявлять различия между группами пассажиров, поэтому его самостоятельный вклад целесообразно проверить на этапе моделирования._

## DF @titanic `same_importance_relatives`.

_Сохраняется как исходная характеристика семейных связей. Анализ семейной структуры показал, что наличие родственников связано с различиями в выживаемости, однако более агрегированные признаки не обязательно полностью заменяют информацию о конкретных родственных связях._

## DF @titanic `high_importance_relatives`.

_Сохраняется по той же причине, но дополнительно отражает наличие близких родственников. Признак использовался в исследовании семейной структуры и показал связь с формированием групп, поэтому его окончательная избыточность должна определяться уже на этапе моделирования._

## DF @titanic `embarked`.

_Сохраняется несмотря на слабую самостоятельную связь с выживаемостью. EDA показал, что значительная часть обнаруженной зависимости объясняется различным распределением социальных категорий между портами. Однако признак не был удалён окончательно, поскольку его дополнительная информация может проявиться во взаимодействии с другими признаками и должна быть проверена моделью._

## DF @titanic `age`


<div class="border" style="border: 2px solid black;width: 99%;height: auto;margin-top: 20px;border-bottom: none;"> <p class="legend" style="margin-top: -8px;margin-left: 10px;padding-left: 15px;background: white;width: 220px;font-weight: 600;">Устранение пропусков</p></div>


_Анализ показал, что отдельные признаки `social_role`, `social_role_and_sex` и `category_significance` недостаточно точно описывают возраст пассажира. Наилучший результат показало совместное использование признаков `category_significance`, `family_type` и `social_role`._

_Заполнение пропущенных значений выполняется медианным возрастом внутри групп, сформированных по этим признакам. Данный подход показал наименьшую среднюю абсолютную ошибку среди рассмотренных стратегий (MAE ≈ 7.55 года)._

In [10]:
titanic = AdvancedProcessingTransformer().fill_missing_age_direct(titanic)

<div class="border" style="border: 2px solid black;width: 99%;height: 10px;border-top: none;"></div>


_Сохраняется, поскольку различия в выживаемости наблюдались не только между полами, но и между возрастными группами. Кроме того, возраст оказался связан с семейной структурой и социальными ролями, поэтому его удаление привело бы к потере самостоятельной информации о жизненном этапе пассажира._

## DF @titanic `name`.

<div class="border" style="border: 2px solid black;width: 99%;height: auto;margin-top: 20px;border-bottom: none;"> <p class="legend" style="margin-top: -8px;margin-left: 10px;padding-left: 15px;background: white;width: 180px;font-weight: 600;">Удаление признака</p></div>


_Необходимые производные уже извлечены и использованы. Сам по себе признак не несёт информации._

In [11]:
titanic = AdvancedProcessingTransformer.remove_name(titanic)

<div class="border" style="border: 2px solid black;width: 99%;height: 10px;border-top: none;"></div>


## DF @titanic `ticket_id`.

<div class="border" style="border: 2px solid black;width: 99%;height: auto;margin-top: 20px;border-bottom: none;"> <p class="legend" style="margin-top: -8px;margin-left: 10px;padding-left: 15px;background: white;width: 180px;font-weight: 600;">Удаление признака</p></div>


_Мы уже исследовали обе составляющие. `ticket_prefix` не дал достаточно самостоятельной информации, а `ticket_number` нужен был прежде всего как технический ключ для группировки. Поэтому само значение `ticket_id`, как категориальный признак модели нам не нужно._

_Если мы оставляем `ticket_id` вместе с `ticket_group_size`, то фактически передаём модели исходный идентификатор группы плюс уже извлечённую из него характеристику. Это выглядит как лишняя информация и потенциально создаёт проблему с обобщением._

In [12]:
titanic = AdvancedProcessingTransformer.remove_ticket_id(titanic)

<div class="border" style="border: 2px solid black;width: 99%;height: 10px;border-top: none;"></div>


In [13]:
titanic = AdvancedProcessingTransformer.optimize_dtypes(titanic)

titanic.info()

<class 'pandas.DataFrame'>
RangeIndex: 887 entries, 0 to 886
Data columns (total 15 columns):
 #   Column                     Non-Null Count  Dtype   
---  ------                     --------------  -----   
 0   passenger_id               887 non-null    uint16  
 1   sex                        887 non-null    category
 2   age                        887 non-null    float32 
 3   same_importance_relatives  887 non-null    uint8   
 4   high_importance_relatives  887 non-null    uint8   
 5   category_significance      887 non-null    category
 6   fare                       887 non-null    float64 
 7   embarked                   887 non-null    category
 8   survived                   887 non-null    int8    
 9   family_size                887 non-null    uint8   
 10  female_count               887 non-null    uint8   
 11  male_count                 887 non-null    uint8   
 12  child_count                887 non-null    uint8   
 13  family_type                887 non-null    cat